# Polarkoordinaten

Wir wollen unser Wissen über die trigonometrischen Funktionen und Polarkoordinaten nutzen, um eine Drehfunktion für Punkte zu entwickeln, die sich anschliessend mit Hilfe einer Plotly-App darstellen lässt.

## Laden der math-Bibliothek
Zunächst laden wir die math-Bibliothek von Python:

In [1]:
import math

Die math-Bibliothek enthält mathematische Funktionen sowie die Konstante $\pi$:
- Konstante $\pi$: `math.pi`
- trigonometrische Funktionen: `math.sin`, `math.cos`, `math.tan`
- Arcusfunktionen: `math.asin`, `math.acos`, `math.atan`
- Wurzelfunktion: `math.sqrt`

Wichtig ist ausserdem zu wissen, dass man $x^2$ als `x**2` eingeben muss.

In [2]:
# Wer will, kann hier etwas herumspielen. (Das hier ist ein Kommentar!)




## Funktionen zur Umrechnung von Gradmass - Bogenmass

Die Funktionen aus der math-Bibliothek arbeiten mit dem Bogenmass. Für Anwender ist es aber bequemer, mit dem Gradmass zu arbeiten. Deshalb schreiben wir eine erste Funktion.

Eine **Funktion** hat einen Namen, nimmt eine Eingabe und liefert eine Ausgabe.
- Unsere Funktion heisst `Bogenmass`.
- Der Eingabewert heisst `alpha` und repräsentiert den Winkel in Grad.
- Der Ausgabewert heisst `phi` und repräsentiert den Winkel im Bogenmass.

In [3]:
def Bogenmass(alpha):
    phi = 2*math.pi/360 * alpha
    return phi

Wir wollen die Funktion testen:

In [4]:
Bogenmass(90)

1.5707963267948966

Der Vollständigkeit halber schreiben wir auch eine Funktion `Gradmass`, die einen Winkel im Bogenmass ins Gradmass umwandelt.

In [5]:
def Gradmass(phi):
    alpha = 360/(2*math.pi) * phi
    return alpha

In [6]:
# Zum Testen


## Funktionen zur Umrechnung von kartesischen Koordinaten - Polarkoordinaten
Funktionen können in Python auch Wertepaare als Ein- oder Ausgabe verarbeiten. Das nutzen wir nun, um Polarkoordinaten in kartesische Koordinaten umzurechnen.

Die Funktion heisst `kartKoordinate`, die Eingabe ist ein Wertepaar `(r, phi)`, die Ausgabe ist `(x, y)`.

In [7]:
def kartKoordinate(r, phi):
    x = r * math.cos(phi)
    y = r * math.sin(phi)
    return (x, y)

In [8]:
# Zum Testen


Die Berechnung von Polarkoordinaten aus kartesischen Koordinaten ist etwas komplizierter. Die Funktion soll `Polarkoordinate` heissen, die Eingabe ist nun ein Wertepaar `(x, y)`, die Ausgabe ist `(r, phi)`.

Allerdings wissen wir, dass die Arcusfunktionen nicht immer den gewünschten Winkel $0\le\varphi\le2\pi$ liefern. Das funktioniert in Python mit `if-elif-else`
```
if Bedingung1:
    Ergebnis1
elif Bedingung2:
    Ergebnis2
else:
    Ergebnis3
```

In [9]:
def Polarkoordinate(x, y):
    r = math.sqrt(x**2+y**2)

    # Fallunterscheidung für phi
    if x==0 and y==0:
        phi = 0.0
    elif x==0 and y>0:
        phi = math.pi/2
    elif x==0 and y<0:
        phi = 3*math.pi/2
    elif x<0:
        phi = math.atan(y/x) + math.pi
    elif y<0:
        phi = math.atan(y/x) + 2*math.pi
    else:
        phi = math.atan(y/x)
    return (r, phi)

In [10]:
# Zum Testen


## Funktion zum Drehen eines Punkts um den Ursprung

Nun benötigen wir noch eine Funktion `Drehung`, die die kartesischen Koordinaten und den Drehwinkel in Grad `(x ,y, alpha)` als Eingabe nimmt und die neuen kartesischen Koordinaten als Ausgabe liefert. Dabei hilft der Umweg über Polarkoordinaten (warum?).

In [11]:
def Drehung(x, y, alpha):
    (r,phi) = Polarkoordinate(x,y)
    (xneu, yneu) = kartKoordinate(r, phi + Bogenmass(alpha))
    return (xneu, yneu)

In [12]:
# Zum Testen


## Darstellung in einer App

Die App ist bereits startklar. Die Liste der Punkte ganz oben kann und soll verändert werden.

In [13]:
Punkte =[(-1,1), (0,1), (0,0), (-1,0)]




# ----------------------- Ab hier nichts mehr ändern ---------------------------------------------------

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'

fig = go.Figure()
max_value = 0
for alpha in range(0, 360):
    gedrehtePunkte = [Drehung(x, y, alpha) for (x,y) in Punkte]      # hier wird Drehung(x, y, alpha) aufgerufen
    x_coords, y_coords = zip(*gedrehtePunkte)
    max_value = max(max_value, max(x_coords), max(y_coords))
    fig.add_trace(
        go.Scatter(
            visible=False,
            x=x_coords,
            y=y_coords,
            mode="lines+markers"
        )
    )
    
fig.data[0].visible = True
max_value *= 1.1

steps = []
for i in range(len(fig.data)):
    step = dict(
        method="update",
        args=[{"visible": [False] * len(fig.data)}],
        label=str(i)
    )
    step["args"][0]["visible"][i] = True
    steps.append(step)

sliders = [dict(
    active=0,
    currentvalue={"prefix": "Wert für alpha: "},
    pad={"t": 50},
    steps=steps
)]

fig.update_layout(
    sliders=sliders,
    xaxis_range=[-max_value, max_value], xaxis_title="X-Achse",
    yaxis_range=[-max_value, max_value], yaxis_title="Y-Achse",
    autosize=False,
    width=800, height=800,
)

fig.show()

## Fleissaufgabe

Überlegen Sie sich, wie man eine Drehung um einen beliebigen Punkt realisieren könnte und passen Sie den Code an.

In [14]:
# Es wird um den Punkt (x_drehpunkt, y_drehpunkt) gedreht.

def Drehung_um_Punkt(x, y, alpha, x_drehpunkt, y_drehpunkt):
    # Man berechnet Radius und Drehwinkel bzgl. des Drehpunkts.
    (r,phi) = Polarkoordinate(x-x_drehpunkt, y-y_drehpunkt)

    # Man führt wie zuvor die Drehung aus und erhält als Zwischenergebnis die (x,y)-Koordinate bzgl. des Drehpunkts.
    (xneu, yneu) = kartKoordinate(r, phi + Bogenmass(alpha))

    # Das Zwischenergebnis muss noch korrigiert werden.
    return (xneu+x_drehpunkt, yneu+y_drehpunkt)

In [15]:
# Hier gibt man die Punkte und den Drehpunkt ein.
Punkte =[(-1,1), (0,1), (0,0), (-1,0)]
Drehpunkt = (-0.5,0.5)


# Die App wurde von oben per Copy-Paste kopiert. Nur der Aufruf der Drehfunktion muss angepasst werden. 

fig = go.Figure()
max_value = 0
for alpha in range(0, 360):
    (x_dreh, y_dreh) = Drehpunkt
    gedrehtePunkte = [Drehung_um_Punkt(x, y, alpha, x_dreh, y_dreh) for (x,y) in Punkte]   # hier wird Drehung_um_Punkt(x, y, alpha, x_drehpunkt, y_drehpunkt) aufgerufen
    x_coords, y_coords = zip(*gedrehtePunkte)
    max_value = max(max_value, max(x_coords), max(y_coords))
    fig.add_trace(
        go.Scatter(
            visible=False,
            x=x_coords,
            y=y_coords,
            mode="lines+markers"
        )
    )
    
fig.data[0].visible = True
max_value *= 1.1

steps = []
for i in range(len(fig.data)):
    step = dict(
        method="update",
        args=[{"visible": [False] * len(fig.data)}],
        label=str(i)
    )
    step["args"][0]["visible"][i] = True
    steps.append(step)

sliders = [dict(
    active=0,
    currentvalue={"prefix": "Wert für alpha: "},
    pad={"t": 50},
    steps=steps
)]

fig.update_layout(
    sliders=sliders,
    xaxis_range=[-max_value, max_value], xaxis_title="X-Achse",
    yaxis_range=[-max_value, max_value], yaxis_title="Y-Achse",
    autosize=False,
    width=800, height=800,
)

fig.show()